# SAM3 fine-tuning on SageMaker TrainingJobs

This notebook launches a SageMaker TrainingJob that fine-tunes SAM3 on
AWS_SAM using the recipe in `sam3/train/configs/aws_sam/aws_sam_finetune.yaml`.

**Inputs** (S3 channels passed to the job):

- `train`  → contains both `AWS_SAM/` (images + LabelMe JSONs) and
  `AWS_SAM_split/` (COCO train.json / test.json from `prepare_aws_sam.py`)
- `pretrained` → contains `sam3.pt` (the local SAM3 weights)

**Outputs**:

- `s3://<bucket>/<prefix>/output/model.tar.gz` — contains
  `checkpoint_merged.pt` (the self-contained fine-tuned checkpoint) +
  the BPE vocab. Ready to download and pass to eval / SageMaker deploy.

## Prerequisites

```bash
pip install sagemaker boto3
aws configure   # or assume an IAM role with sagemaker:* + s3 access
```

Run this notebook on a machine that has the SAM3 source tree (so the
SDK can package it) and AWS credentials. A SageMaker notebook
instance or your laptop both work.

In [ ]:
# Pin sagemaker<3 — the v3 SDK (sagemaker>=3.0) is a major rewrite that
# moves PyTorch estimator + Session + get_execution_role out of their
# canonical paths. This notebook targets the long-stable v2 API.
%pip install --quiet 'sagemaker>=2.230,<3' boto3
# IMPORTANT: after installing, restart the kernel before running the
# next cell — otherwise the previously-imported v3 module stays cached.


In [1]:
# ---------- Configuration: edit these ----------
import os
import sagemaker
import boto3

# Sanity-check SDK version. The v3 SDK has a different layout and this
# notebook is written against v2. The cell above pins sagemaker<3.
_v = getattr(sagemaker, "__version__", "unknown")
if _v.startswith("3.") or _v.startswith("4."):
    raise RuntimeError(
        f"Detected sagemaker {_v}, but this notebook requires v2.x. "
        f"Run the previous cell, then RESTART THE KERNEL, then retry."
    )
print("sagemaker version:", _v)

# v2-style imports (re-exported at top-level).
from sagemaker import get_execution_role

sess = sagemaker.Session()
role = get_execution_role()
sagemaker_default_bucket = sess.default_bucket()
region = sess.boto_session.region_name
print("sagemaker_default_bucket:", sagemaker_default_bucket)
print("sagemaker_region:", region)

S3_PREFIX        = "projects/sam3/data/aws_sam_finetune_v1"

# Local paths (these get uploaded to s3 once at launch time)
LOCAL_REPO_ROOT  = "/home/ec2-user/SageMaker/efs/Projects/sam3"
LOCAL_DATA       = "/home/ec2-user/SageMaker/efs/Projects/sam3/data"             # contains AWS_SAM/ and AWS_SAM_split/
LOCAL_PRETRAINED = "/home/ec2-user/SageMaker/efs/Models/sam3"                    # contains sam3.pt

# Instance & training settings
INSTANCE_TYPE    = "ml.p4de.24xlarge"     # 8 × A100 80GB. Use ml.p5.48xlarge for 8 × H100.
INSTANCE_COUNT   = 1                     # multi-node not wired up yet
NUM_GPUS         = 8                     # per-instance
MAX_RUNTIME_S    = 24 * 3600
VOLUME_SIZE_GB   = 500

# Hyperparameter overrides (None = use yaml defaults)
HP = {
    "num-gpus":           NUM_GPUS,
    # Optional — set/uncomment to override config defaults:
    # "max-epochs":         20,
    # "train-batch-size":   16,
    # "lr-scale":           0.1,
    # "num-train-workers":  0,
}

# Job name (timestamp will be appended automatically by the SDK)
JOB_NAME_PREFIX = "sam3-aws-sam-ft"

print("repo root:", LOCAL_REPO_ROOT)


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ubuntu/.config/sagemaker/config.yaml
sagemaker version: 2.257.3
sagemaker_default_bucket: sagemaker-us-west-2-452145973879
sagemaker_region: us-west-2
repo root: /home/ec2-user/SageMaker/efs/Projects/sam3


## 1. Upload data + pretrained weights to S3

The training job needs the dataset and pretrained checkpoint pre-staged
in S3. This step is **one-time** — subsequent runs reuse the same S3 paths.

In [2]:
import boto3
import sagemaker
from sagemaker.s3 import S3Uploader

boto_session = boto3.Session(region_name=region)
sagemaker_session = sagemaker.Session(boto_session=boto_session)

s3_train_uri      = f"s3://{sagemaker_default_bucket}/{S3_PREFIX}/input/train"
s3_pretrained_uri = f"s3://{sagemaker_default_bucket}/{S3_PREFIX}/input/pretrained"
s3_output_uri     = f"s3://{sagemaker_default_bucket}/{S3_PREFIX}/output"

print("train data    →", s3_train_uri)
print("pretrained    →", s3_pretrained_uri)
print("output        →", s3_output_uri)

train data    → s3://sagemaker-us-west-2-452145973879/projects/sam3/data/aws_sam_finetune_v1/input/train
pretrained    → s3://sagemaker-us-west-2-452145973879/projects/sam3/data/aws_sam_finetune_v1/input/pretrained
output        → s3://sagemaker-us-west-2-452145973879/projects/sam3/data/aws_sam_finetune_v1/output


In [ ]:
# Upload the dataset (AWS_SAM/ + AWS_SAM_split/). Comment out after the
# first successful upload — the data is large.
S3Uploader.upload(
    local_path=LOCAL_DATA,
    desired_s3_uri=s3_train_uri,
    sagemaker_session=sagemaker_session,
)
print("✓ data uploaded")

In [ ]:
# Upload the pretrained checkpoint (~9 GB). Comment out after the
# first successful upload.
S3Uploader.upload(
    local_path=LOCAL_PRETRAINED,
    desired_s3_uri=s3_pretrained_uri,
    sagemaker_session=sagemaker_session,
)
print("✓ pretrained uploaded")

## 2. Launch the TrainingJob

SageMaker's PyTorch estimator packages the local source tree (the
sam3 repo) and ships it to the container, then runs `train_entry.py`
as the entry point. The script installs deps, patches the config to
use SageMaker channel paths, runs `sam3/train/train.py`, then merges
the resulting checkpoint and writes it to `/opt/ml/model/`.

In [ ]:
# Build a slim source-bundle directory that the SageMaker PyTorch
# estimator will actually upload.
#
# WHY: SageMaker SDK v2 does NOT honor .sagemakerignore (that's v3-only).
# If we pass source_dir=LOCAL_REPO_ROOT, the SDK tarballs the ENTIRE
# repo including data/ (multi-GB) and runs/ (9GB checkpoints each),
# which silently hangs the upload for tens of minutes and then aborts
# with an opaque error (fit() returns None instead of a job handle).
#
# Instead, we assemble a small staging dir with just the code we need
# and point source_dir at that. The staging dir uses SYMLINKS to avoid
# copying gigabytes of package assets — the SDK follows symlinks when
# it builds the tarball.
import os
import shutil

STAGING_DIR = os.path.expanduser("~/sam3_sagemaker_staging")

# Wipe any previous staging.
if os.path.exists(STAGING_DIR):
    shutil.rmtree(STAGING_DIR)
os.makedirs(STAGING_DIR)

# Include: sam3/ package, scripts/ (has train_entry.py), pyproject.toml,
# MANIFEST.in, README.md, .git-related files needed by setuptools_scm
# (only .git-archive-info if it exists). We copy small files, symlink
# the sam3 package tree (it has ~1.4 MB of BPE vocab that must ship).
INCLUDES = [
    "sam3",                # the actual package (source + assets/bpe*.txt.gz)
    "scripts",             # has scripts/finetune/sagemaker/train_entry.py + merge_checkpoint.py
    "pyproject.toml",
    "MANIFEST.in",
    "README.md",
    "LICENSE",
]
for name in INCLUDES:
    src = os.path.join(LOCAL_REPO_ROOT, name)
    dst = os.path.join(STAGING_DIR, name)
    if not os.path.exists(src):
        print(f"  skip (missing): {name}")
        continue
    if os.path.isdir(src):
        # Copy the tree, but skip caches and other cruft. Symlinks
        # would be simpler but the SDK's tarball packager sometimes
        # doesn't follow them, so we do a filtered copy instead.
        shutil.copytree(
            src, dst,
            ignore=shutil.ignore_patterns(
                "__pycache__", "*.pyc", "*.pyo",
                ".ipynb_checkpoints", ".pytest_cache", ".mypy_cache",
                "*.egg-info", "build", "dist",
            ),
        )
    else:
        shutil.copy2(src, dst)

# Verify staging size.
total = 0
for root, dirs, files in os.walk(STAGING_DIR):
    for f in files:
        total += os.path.getsize(os.path.join(root, f))
print(f"staging dir: {STAGING_DIR}")
print(f"staging size: {total / 1e6:.1f} MB")

# From here forward, source_dir points at the staging dir, not the repo.
LOCAL_REPO_ROOT_FOR_SAGEMAKER = STAGING_DIR
print(f"\nsource_dir for SageMaker: {LOCAL_REPO_ROOT_FOR_SAGEMAKER}")


In [ ]:
from sagemaker.pytorch import PyTorch
from sagemaker.inputs import TrainingInput

estimator = PyTorch(
    entry_point="train_entry.py",
    # Use the STAGING dir, not the full repo — SDK v2 doesn't honor
    # .sagemakerignore, so pointing at the repo root uploads data/,
    # runs/, .git/, etc. and hangs. The previous cell built a slim
    # copy at ~/sam3_sagemaker_staging with just what we need.
    source_dir=LOCAL_REPO_ROOT_FOR_SAGEMAKER,
    role=role,
    framework_version="2.4.0",                        # cu124 base image
    py_version="py311",
    instance_type=INSTANCE_TYPE,
    instance_count=INSTANCE_COUNT,
    volume_size=VOLUME_SIZE_GB,
    max_run=MAX_RUNTIME_S,
    output_path=s3_output_uri,
    base_job_name=JOB_NAME_PREFIX,
    sagemaker_session=sagemaker_session,
    hyperparameters=HP,                                # forwarded as CLI args
    environment={
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        # Tells train_entry.py where to find its own script. SageMaker
        # by default chdirs into /opt/ml/code/ which contains the
        # repo; the entry script lives at the same path it has in
        # the repo so this is mostly informational.
        "SAM3_REPO_ROOT": "/opt/ml/code",
    },
    disable_profiler=True,
    # Tip: set this if you want fast restart on interruption.
    keep_alive_period_in_seconds=1800,
)

# The SDK looks for source_dir/entry_point. Because train_entry.py
# lives under scripts/finetune/sagemaker/ inside the staging dir,
# tell the SDK to look there:
estimator.entry_point = "scripts/finetune/sagemaker/train_entry.py"

print("Estimator built. ImageURI:", estimator.training_image_uri())


In [ ]:
# import boto3
# sm = boto3.client("sagemaker", region_name=region)
# resp = sm.list_training_jobs(
#     NameContains=JOB_NAME_PREFIX, MaxResults=5,
#     SortBy="CreationTime", SortOrder="Descending",
# )
# for j in resp["TrainingJobSummaries"]:
#     print(j["TrainingJobName"], "→", j["TrainingJobStatus"])

In [6]:
# Kick it off. estimator.fit() is blocking — pass wait=False if you
# want the cell to return immediately and monitor via the SageMaker
# console.
estimator.fit(
    inputs={
        "train":      TrainingInput(s3_data=s3_train_uri),
        "pretrained": TrainingInput(s3_data=s3_pretrained_uri),
    },
    wait=False,        # set False to background the job
    logs="All",       # stream all container logs to this notebook
)
print("Job name:", estimator.latest_training_job.job_name)
print("Model artifact:", estimator.model_data)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:12                                                                                   │
│                                                                                                  │
│    9 │   wait=False,        # set False to background the job                                    │
│   10 │   logs="All",       # stream all container logs to this notebook                          │
│   11 )                                                                                           │
│ ❱ 12 print("Job name:", estimator.latest_training_job.job_name)                                  │
│   13 print("Model artifact:", estimator.model_data)                                              │
│   14                                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
AttributeError: 'NoneType' object has no attribute 'job_name'

In [ ]:
# Fetch CloudWatch logs for a training job. Handy for debugging failed jobs.
import boto3
logs = boto3.client("logs", region_name=region)

job_name = "sam3-aws-sam-ft-2026-06-30-13-30-41-376"   # ← edit to the job you want to inspect
log_group = "/aws/sagemaker/TrainingJobs"

# CloudWatch doesn't allow orderBy=LastEventTime WITH a logStreamNamePrefix
# — use LogStreamName ordering (or just list without prefix).
streams = logs.describe_log_streams(
    logGroupName=log_group,
    logStreamNamePrefix=job_name,
    orderBy="LogStreamName",
    descending=True,
    limit=5,
)["logStreams"]
print("streams:", [s["logStreamName"] for s in streams])
if not streams:
    print("(no streams — job hasn't started yet, or was terminated before writing logs)")

# Grab the last ~200 lines from each stream
for s in streams:
    print("\n===", s["logStreamName"], "===")
    events = logs.get_log_events(
        logGroupName=log_group,
        logStreamName=s["logStreamName"],
        startFromHead=False,
        limit=200,
    )["events"]
    for e in events[-100:]:
        print(e["message"].rstrip())


## 3. Download the merged checkpoint back to local disk

SageMaker uploads anything written to `/opt/ml/model/` as a
`model.tar.gz` in the output S3 path. `train_entry.py` writes
`checkpoint_merged.pt` (the self-contained fine-tuned weights) +
the BPE vocab there.

In [ ]:
from sagemaker.s3 import S3Downloader

# estimator.model_data is the s3 URI of model.tar.gz
local_artifact = "runs/sagemaker_artifact"
os.makedirs(local_artifact, exist_ok=True)
S3Downloader.download(
    s3_uri=estimator.model_data,
    local_path=local_artifact,
    sagemaker_session=sagemaker_session,
)
print("downloaded to:", local_artifact)

# Unpack
import tarfile
tar_path = os.path.join(local_artifact, "model.tar.gz")
with tarfile.open(tar_path) as t:
    t.extractall(local_artifact)
print("contents:", os.listdir(local_artifact))

## 4. Run eval against the downloaded checkpoint

The merged file is identical to what `merge_checkpoint.py` would
produce locally, so you can plug it straight into the eval harness:

```bash
python scripts/finetune/eval/evaluate_interactive.py \
    --coco data/AWS_SAM_split/test.json \
    --image-root data/AWS_SAM \
    --checkpoint runs/sagemaker_artifact/checkpoint_merged.pt \
    --output runs/eval/finetuned_sagemaker.json
```

## Notes / gotchas

- **First run is slow** because `pip install -e ".[dev,train]"` runs
  inside the container. If you do many runs, consider building a
  custom Docker image that has SAM3 pre-installed and pass it via
  `image_uri=...` on the estimator.
- **S3 charges and bandwidth**: data + pretrained ≈ 10 GB, model
  artifact ≈ 9 GB. Cleanup with `aws s3 rm --recursive` when done.
- **Spot training**: add `use_spot_instances=True, max_wait=...` to
  the estimator to drop costs ~60% on long jobs at the risk of
  preemption. SageMaker auto-resumes from the trainer's last
  checkpoint on the next replica.
- **Multi-instance**: set `instance_count > 1` only after you've
  wired up the trainer's distributed init for multi-node SLURM-less
  launch. The current trainer uses single-node-multi-GPU via
  `mp.spawn`; multi-node SageMaker would need the `submitit` path
  or a `torchrun`-style wrapper around `train_entry.py`.